In [1]:
import pandas as pd
from zipfile import ZipFile

In [3]:
import os

os.chdir(r"D:\Proj\Portfolio\projects\01-feedback-loop-corruption")

In [5]:
identity_zip_path = "data/train_identity.csv.zip"
txn_zip_path = "data/train_transaction.csv.zip"


with ZipFile(identity_zip_path) as identity_zip:
    with identity_zip.open("train_identity.csv") as identity_file:
        identity_df = pd.read_csv(identity_file)
        identity_df.to_csv("data/identity.csv", index=False)

with ZipFile(txn_zip_path) as txn_zip:
    with txn_zip.open("train_transaction.csv") as txn_file:
        txn_df = pd.read_csv(txn_file)
        txn_df.to_csv("data/transaction.csv", index=False)

In [11]:
import shutil
# os.mkdir("data/processed")
shutil.move("data/identity.csv", "data/processed/identity.csv")
shutil.move("data/transaction.csv", "data/processed/transaction.csv")
os.mkdir("data/raw")
shutil.move("data/train_identity.csv.zip", "data/raw/train_identity.csv.zip")
shutil.move("data/train_transaction.csv.zip", "data/raw/train_transaction.csv.zip")

'data/raw/train_transaction.csv.zip'

In [7]:
len(identity_df)/len(txn_df)

0.2442391709283029

In [16]:
txn_df[txn_df['isFraud'] == 1].count()

TransactionID     20663
isFraud           20663
TransactionDT     20663
TransactionAmt    20663
ProductCD         20663
                  ...  
V335               3686
V336               3686
V337               3686
V338               3686
V339               3686
Length: 394, dtype: int64

## EDA

Questions: what columns exist and how empty are they · how much fraud · does fraud change over time · does having identity info matter · how amounts differ · which columns are worth keeping.

Sections marked **YOUR CELL** are yours to write.

In [ ]:
# 1. Shapes and column groups. Columns are anonymised, so the name prefix is the group (C1..C14 -> "C").
import re

def col_group(c):
    m = re.match(r"([A-Za-z_]+?)_?\d+$", c)
    return m.group(1) if m else c

print("transactions:", txn_df.shape, "| identity:", identity_df.shape)
for name, df in [("transaction", txn_df), ("identity", identity_df)]:
    print(f"\n{name} column groups (group -> number of columns):")
    print(pd.Series([col_group(c) for c in df.columns]).value_counts().to_string())

In [ ]:
# 2. How empty is each column group? (% missing: average over the group's columns, least-empty column, most-empty column)
for name, df in [("transaction", txn_df), ("identity", identity_df)]:
    miss = df.isna().mean().groupby(col_group).agg(["mean", "min", "max"]).mul(100).round(1)
    print(f"\n{name}: % missing per group")
    print(miss.sort_values("mean", ascending=False).to_string())

print("\ntransaction columns more than 90% empty:", int((txn_df.isna().mean() > 0.9).sum()))

### Your sections (3–5)

In [ ]:
# 3. YOUR CELL: how many fraud transactions are there, and what share of all transactions?
# Column: isFraud (1 = fraud). Your earlier cell used .count(), which counts non-empty cells per column —
# that's why V339 showed 3686 instead of 20663. Count rows instead.


In [ ]:
# 4. YOUR CELL: transactions per day and fraud rate per day, over the 182 days.
# TransactionDT = seconds from an unknown start. Turn it into a whole day number (there are 86400 seconds in a day),
# group by that day, compute the count and the fraud rate, and plot both.
# Question to answer: is the fraud rate steady, trending, or jumping around?


In [ ]:
# 5. YOUR CELL: fraud rate for transactions WITH identity info vs WITHOUT.
# A transaction "has identity" if its TransactionID appears in identity_df.
# Question to answer: is the difference big enough that "has identity" is itself a useful signal?


In [ ]:
# 6. TransactionAmt: fraud vs not fraud. Log scale, because a few huge amounts would squash the plot.
import numpy as np
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4))
for label, grp in txn_df.groupby("isFraud"):
    ax.hist(np.log10(grp["TransactionAmt"]), bins=60, density=True, alpha=0.5,
            label="fraud" if label == 1 else "not fraud")
ax.set_xlabel("log10(TransactionAmt)")
ax.set_ylabel("density (each group sums to 1)")
ax.legend()
plt.show()

print(txn_df.groupby("isFraud")["TransactionAmt"].describe().round(2))